In [1]:
from selenium.common.exceptions import NoSuchElementException

import requests
import json
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json
from tqdm import tqdm
import random
import uuid
from threading import Lock

import logging
import datetime
from traceback import print_exc

import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.proxy import Proxy, ProxyType

import os
os.environ['https_proxy'] = "http://proxy.hcm.fpt.vn:80/"
os.environ['http_proxy'] = "http://proxy.hcm.fpt.vn:80/"
os.environ['no_proxy'] = "localhost,127.0.0.0,127.0.1.1,127.0.1.1,local.home"


# Define your proxy
proxy = Proxy()
proxy.proxy_type = ProxyType.MANUAL
proxy.http_proxy = "http://hndc11.proxyxoay.net:62142"
proxy.ssl_proxy = "http://hndc11.proxyxoay.net:62142"


def write_to_file(data, filename='access_token.json'):
    file_lock = Lock()
    with file_lock:
        with open(filename, 'a') as f:
            f.write(json.dumps(data) + '\n')
            
def pass_privacy_notice(driver):
    if "privacynotice" in driver.current_url:
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[type="button"]')))
        driver.find_element(By.CSS_SELECTOR, 'button[type="button"]').click()
    return 0
    
def get_account_passwork():
    with open('resource/account.txt', 'r') as f:
        accounts = f.readlines()
    accounts = [account.replace('\n',"").split('|') for account in accounts]
    return accounts

def random_sleep(min_time, max_time):
    time.sleep(np.random.uniform(min_time, max_time))
    return 0

def get_author(token):
    session = requests.Session()
    paramsGet = {
        "AadObjectId": "",
        "Smtp": 'lionelmessi@gmail.com',
        "OlsPersonaId": "",
        "UserPrincipalName": "",
        "RootCorrelationId": str(uuid.uuid4()),
        "CorrelationId": str(uuid.uuid4()),
        "ClientCorrelationId": str(uuid.uuid4()),
        "PersonaDisplayName": "",
        "UserLocale": "en-US",
        "ExternalPageInstance": "00000000-0000-0000-0000-000000000000",
        "PersonaType": "User"
    }
    headers = {
        "Authorization": token,
        "X-ClientFeature": "LivePersonaCard",
        "Accept": "text/plain, application/json, text/json",
        "X-ClientType": "OwaMail",
        "X-HostAppCapabilities": "{}",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:57.0) Gecko/20100101 Firefox/57.0",
        "Connection": "close",
        "X-LPCVersion": "1.20210418.1.0"
    }

    response = session.get(
        "https://sfnam.loki.delve.office.com/api/v1/linkedin/profiles/full", 
        params=paramsGet, 
        headers=headers
    )
    return response.status_code,response.text


def create_driver(headless=True):
    chrome_options = Options()
    
    if headless:
        chrome_options.add_argument("--headless")
    
    chrome_options.add_argument("--window-size=1920,1200")
    chrome_options.add_argument("--disable-setuid-sandbox")
    chrome_options.add_argument("enable-automation")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--dns-prefetch-disable")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--incognito")
    chrome_options.add_argument('--proxy-server=http://proxy.hcm.fpt.vn:80')
    chrome_options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    
    # # Set up logging preferences
    # caps = DesiredCapabilities.CHROME
    # caps['goog:loggingPrefs'] = {'performance': 'ALL'}  # Enable performance logging
    
    # Create a Service object for ChromeDriver
    s = Service(executable_path='resource/chromedriver')
    
    # Initialize the WebDriver with the service, options, and capabilities
    driver = webdriver.Chrome(service=s, options=chrome_options)
    
    logging.info("Create driver successfully")
    
    driver.implicitly_wait(10)
    return driver



def get_access_token_from_outlook(account, first_name='Lionel', last_name = 'Messi', recipient_email = 'lionelmessi1997@gmail.com'):
    driver_wait_time = 10
    username = account[0]
    password = account[1]
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--headless")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    proxy = "hndc11.proxyxoay.net:62142"
    # options.add_argument(f'--proxy-server=http://{proxy}')
    # add proxy to driver
    driver = create_driver()
    driver.get('https://outlook.office365.com/')
    random_sleep(2,4)
    try:



        # WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, '//div[@id="meControl"]')))
        # driver.find_element(By.XPATH, '//div[@id="meControl"]').click()

        # WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[aria-label="Enter your email, phone, or Skype."]')))
        # email_input = driver.find_element(By.CSS_SELECTOR, 'input[aria-label="Enter your email, phone, or Skype."]')
        # email_input.send_keys(username)
        # driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]').click()
        #
        # WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, f'input[aria-label="Enter the password for {username}"]')))
        # password_input = driver.find_element(By.CSS_SELECTOR, f'input[aria-label="Enter the password for {username}"]')
        # password_input.send_keys(password)
        # driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]').click()
        #
        # pass_privacy_notice(driver)
        #
        # WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[type="submit"]')))
        # driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]').click()

        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="email"]')))
        email_input = driver.find_element(By.CSS_SELECTOR, 'input[type="email"]')
        email_input.send_keys(username, Keys.ENTER)
        random_sleep(2,4)

        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="password"]')))
        password_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
        password_input.send_keys(password, Keys.ENTER)
        random_sleep(2,4)

        pass_privacy_notice(driver)

        try:
            if 'lock' in driver.find_element(By.CSS_SELECTOR, 'div[role="heading"]').text:

                write_to_file({'time':datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                               'account':username,'status_code':800,
                               'response':None,
                               'authorization':None}
                              )
                driver.quit()
                return (200,'Success')
        except:
            pass

        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[type="submit"]')))
        driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]').click()

        # WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[aria-label="Go to Outlook"]')))
        random_sleep(1,2)
        driver.get('https://outlook.live.com/people/')
        random_sleep(2,4)

        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[aria-label="New contact"]')))

        try:
            WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.XPATH, f'//div[contains(@aria-label, "{first_name} {last_name}")]')))
            contact_tab = driver.find_element(By.XPATH, f'//div[contains(@aria-label, "{first_name} {last_name}")]')
            contact_tab.click()
        except:
            driver.find_element(By.CSS_SELECTOR, 'button[aria-label="New contact"]').click()

            WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[aria-label="First name"]')))
            first_name_input = driver.find_element(By.CSS_SELECTOR, 'input[aria-label="First name"]')
            first_name_input.send_keys(first_name)
            last_name_input = driver.find_element(By.CSS_SELECTOR, 'input[aria-label="Last name"]')
            last_name_input.send_keys(last_name)
            email_input = driver.find_element(By.CSS_SELECTOR, 'input[aria-label="Email address 1"]')
            email_input.send_keys(recipient_email)
            driver.find_element(By.CSS_SELECTOR, 'button[data-automation="LPESave"]').click()

            WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.XPATH, f'//div[contains(@aria-label, "{first_name} {last_name}")]')))
            driver.find_element(By.XPATH, f'//div[contains(@aria-label, "{first_name} {last_name}")]').click()

        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[data-content="Overview"]')))
        driver.find_element(By.CSS_SELECTOR, 'button[data-content="Overview"]').click()
        WebDriverWait(driver, driver_wait_time).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[data-log-name="PanelFooter"]')))
        network_data = driver.get_log('performance')
        driver.quit()

        network_data = [data for data in network_data if ('loki.delve.office.com/api/' in data['message'])&('Bearer' in data['message'])]
        token_list = ["Bearer " + data['message'].split('Bearer ')[1].split('"')[0].replace('\\','') for data in network_data]
        token_list = list(set(token_list))
        for token in token_list:
            write_to_file({'time':datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'account':username,'status_code':get_author(token)[0],
                'response':get_author(token)[1],
                'authorization':token}
                )

        # write_to_file({'time':datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        #             'account':username,
        #             'authorization':token_list}
        #             )
        return (200,'Success')
    except Exception as e:
        driver.quit()
        print(f"Failed to get access token for account {account} {print_exc()}")
        return (400, str(e))

def get_token_from_outlook_main(account, first_name='Lionel', last_name = 'Messi', recipient_email = 'lionell_mes@gmail.com'):
    i = 0
    while i < 2:
        try:
            if get_access_token_from_outlook(account, first_name, last_name, recipient_email)[0] == 400:
                # print(f"Failed to get access token for account {account} ")
                i += 1
            else:
                # print(f"Success to get access token for account {account}")
                break
        except Exception as e:
            # print(f"Failed to get access token for account {account} {print_exc()}")
            i += 1
# get_access_token_from_outlook(('cafagnoagq674287@hotmail.com','cocW858bPN832'))

In [8]:
def main():
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(get_token_from_outlook_main, account) for account in get_account_passwork()]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing account"):
            try:
                future.result()
            except Exception as exc:
                print(f'Task generated an exception: {exc}')

### Get token from list outlook account in account file

In [9]:
main()

Processing emails:   2%|▎         | 1/40 [00:10<06:43, 10.36s/it]

Success to get access token for account ['test_crawl_2@outlook.com', '0965584120Dc', 'cradlerhzr373240jx@moakt.cc']


Processing emails:   5%|▌         | 2/40 [00:30<10:03, 15.88s/it]

Success to get access token for account ['test_crawl_3@outlook.com', '0965584120Dc', 'cradlerhzr373240jx@moakt.cc']


Processing emails:   8%|▊         | 3/40 [00:32<05:58,  9.68s/it]

...

Success to get access token for account ['bazeqqc85708@hotmail.com', 'gwK407mpEI461', 'bazeqqc8570851t@moakt.cc']


Processing emails:  98%|█████████▊| 39/40 [06:29<00:07,  7.86s/it]

Success to get access token for account ['mussebro549050@outlook.com', 'rknRJN7p67778', 'mussebro5490502l5@moakt.cc']


Processing emails: 100%|██████████| 40/40 [06:45<00:00, 10.15s/it]

Success to get access token for account ['niemieltds666065@hotmail.com', 'zOEF768mtK725', 'niemieltds666065a9h@moakt.cc']


### Step 1: Get email register linkedin account among emails

In [10]:
import requests
import datetime
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed

file_lock = Lock()

def write_data(data):
    with file_lock:
        with open('email_match_linkedin.json', 'a') as f:
            f.write(json.dumps(data) + '\n')


def get_author_main(smtp, token):
    session = requests.Session()
    paramsGet = {
        "AadObjectId": "",
        "Smtp": smtp,
        "OlsPersonaId": "",
        "UserPrincipalName": "",
        "RootCorrelationId": str(uuid.uuid4()),
        "CorrelationId": str(uuid.uuid4()),
        "ClientCorrelationId": str(uuid.uuid4()),
        "PersonaDisplayName": "",
        "UserLocale": "en-US",
        "ExternalPageInstance": "00000000-0000-0000-0000-000000000000",
        "PersonaType": "User"
    }
    headers = {
        "Authorization": token,
        "X-ClientFeature": "LivePersonaCard",
        "Accept": "text/plain, application/json, text/json",
        "X-ClientType": "OwaMail",
        "X-HostAppCapabilities": "{}",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:57.0) Gecko/20100101 Firefox/57.0",
        "Connection": "close",
        "X-LPCVersion": "1.20210418.1.0"
    }

    response = session.get(
        "https://sfnam.loki.delve.office.com/api/v1/linkedin/profiles/full",
        params=paramsGet,
        headers=headers
    )

    write_data({'time': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 'token': token, 'email': smtp,
                "status": response.status_code,
                "response": response.text,})
    return 0

In [11]:
df_tokens = pd.read_json('access_token.json', lines=True)
df_tokens = df_tokens[(df_tokens['time'].str.contains(datetime.datetime.now().strftime("%Y-%m-%d")))].sort_values('time').drop_duplicates('account',keep='last')
df_tokens = df_tokens[df_tokens['status_code']==200]
df_tokens.head()

,time,account,status_code,response,authorization
1345,2024-08-14 15:19:27,test_crawl_3@outlook.com,200.0,"{""resultTemplate"":""SearchResult"",""bound"":false...",Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...
1348,2024-08-14 15:19:58,test_crawl_6@outlook.com,200.0,"{""resultTemplate"":""SearchResult"",""bound"":false...",Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...
1349,2024-08-14 15:20:02,test_crawl_7@outlook.com,200.0,"{""resultTemplate"":""SearchResult"",""bound"":false...",Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...
1352,2024-08-14 15:20:38,cafagnoagq674287@hotmail.com,200.0,"{""resultTemplate"":""SearchResult"",""bound"":false...",Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...
1353,2024-08-14 15:20:44,cradlerhzr37324@hotmail.com,200.0,"{""resultTemplate"":""SearchResult"",""bound"":false...",Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...


In [12]:
df_emails = pd.read_excel('resource/email.xlsx')
df_emails.head()

,email,group
0,t@fpt.vn,fpt.vn
1,htsteelvn@yahoo.com,yahoo.com
2,noble_z4@yahoo.com,yahoo.com
3,tung_tkc@yahoo.com,yahoo.com
4,nhungdlh@yahoo.com,yahoo.com


In [13]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import itertools

# Assuming df_tokens and df_emails are already defined
tokens = df_tokens['authorization'].tolist()
emails = df_emails['email'].tolist()

# Function to get token in a round-robin fashion
def get_token(token_list):
    token_cycle = itertools.cycle(token_list)  # Cycle through tokens endlessly
    while True:
        yield next(token_cycle)

In [14]:
# Track total progress
total_tasks = len(emails)

# Create a tqdm progress bar
progress_bar = tqdm(total=total_tasks, desc="Processing emails", unit="email")

# Initialize token generator
token_generator = get_token(tokens)

for i_e in range(0, len(emails), 800):
    with ThreadPoolExecutor(max_workers=800) as executor:
        futures = [
            executor.submit(get_author_main, emails[i], next(token_generator)) 
            for i in range(i_e, min(i_e + 800, len(emails)))
        ]
        for future in as_completed(futures):
            future.result()
            progress_bar.update(1)

progress_bar.close()
print("All tasks completed.")


Processing emails: 100%|██████████| 3000/3000 [01:43<00:00, 28.96email/s] 

All tasks completed.


#### Filter email having linkedin account

In [18]:
df_result = pd.read_json('email_match_linkedin.json', lines=True)
df_result = df_result[df_result['status']==200]
df_result['response'] = df_result['response'].apply(lambda x: json.loads(x))
df_result['person'] = df_result['response'].apply(lambda x: len(x.get('persons', None)))
df_match_linkedin = df_result[df_result['person']>0]
df_match_linkedin = df_match_linkedin.drop_duplicates('email')
df_match_linkedin.head()

,time,token,email,status,response,person
3012,2024-08-13 10:46:35,Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...,duongnh@yahoo.com,200,"{'resultTemplate': 'SearchResult', 'bound': Fa...",1
3015,2024-08-13 10:46:35,Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...,tuanquoc230968@yahoo.com,200,"{'resultTemplate': 'SearchResult', 'bound': Fa...",1
3016,2024-08-13 10:46:35,Bearer EwBIA92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...,sunco26@gmail.com,200,"{'resultTemplate': 'SearchResult', 'bound': Fa...",1
3068,2024-08-13 10:46:35,Bearer EwA4A92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...,vinhmarketing@gmail.com,200,"{'resultTemplate': 'SearchResult', 'bound': Fa...",1
3079,2024-08-13 10:46:35,Bearer EwAoA92CBAAU0ONDFrG0B7oPr/VdyzjriN+c3vg...,tinnc2003@yahoo.com,200,"{'resultTemplate': 'SearchResult', 'bound': Fa...",1


### Step 2: Get profile linkedin with email registered linkedin account

- tokens : take from outlook account linked linkedin account or outlook account belongs business
- emails: email is used to register linkedin account, be filtered from step 1

In [52]:

tokens = ["Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjhDQkMtSFMyNTYiLCJ4NXQiOiJtRXpHeF80OE9MOWcxdXh1QkFnRXVOdGJqcVEiLCJ6aXAiOiJERUYifQ.naoC0oeqX_b6GIn1mCWxkKjqU_8es73oleKC2U9R3QehgXNlaUyHV-r_I-DOR6bRLO3RSYelGdVK3ilS3ij8zxWtMOVSW1IlsMfruyfpw3cTsHdapiSBn8D-6W1PClAxQKfWd009LmaiGW8D84AIHYboDs0spVI6dLe7KMxXs80X4CRhNTAbsXjnFSDQgFDgzo-HaF5Q-OkIeJTl4D9_dVbxW8n8pcYWfldZhuRdjCy0_WKJ6dxzX_kjHCgne9oPsy9QPJOW_3b9tuQ4s-7pb5ExFwS3rOwbDTMIjo3lzQN8qmnZ4RHZ6rq0pT-ZUCqHoNCkcYAFmjyey9qAIb_OBA.4ERGWgsYLTrLLqWWTZT4bg.WHylV2O7RAa8iPQq6DNjVdAl2ST741hxzK2sJug-HPyxe1C_tsz3zwDQQDuukeFawt0XnsK_4W1ld3MrkDlNUg6uPHFtOLudjMyr2RHjzJ5Ik5oVKQcWAVPUmsLE_GD4002YYbbEEBjO71rBPi9FnRwSst8oegnatiZu5Spsdc4PUG05GnA2VatrgcHS4C2F-JC5TciVFuMherRwbxXrnqAjIuB57tAhOomdTmfzT4X-vITnzu-UmN_3hrAU6GT4KVzIlBRc6VAR9LtlAB6NQHAdLsoCJGFc6qbu_rSVYszT2Kr0IMCobIpFsY8oHZSuetTUKnlHgbVPrg1OXoXQBPW-0zt0rL2wCoPAWKJBJbLuvtp7BXZG6_QOnT0-obMQIafz8T2T8PP9w3c_F3-UINcZpzOchDRTZR1X43uhvK9nal8TrApwIGU9jIzckellqCozu-pO2GesCmTrr9hbVK9_xB5dJ2F89aNCQS-vug4HKhIDgCRajFc6d-ZXyRNME-lWjDEpq1Y8S1ZHLaYaryfBX8TQ_EmneuKNt8pr_7TKJdnaSnJGap8PvG9dzqXrym7t4GfDBTmOOdbItIS3BIEMJ2lbe02HFfZkWpRRkyheWCI-Tdpye11NrfFRGb3y8wwjwV9FLMET0n5aMCowYRizjVMdcrp6qLHBa74N8KNBq1Fu0RjKo7Rq-iGQYUAviERdMRRE3PQAq68qi-5r0f68cN_sDCcIqKHoRwT1foQwed1ug5X5_G0Xyzdh8Jf9f0F2zv11eReaQjeCwOAVR08nD1AyAPSK0ulojODdtPwTemx6vBkwA39VKsm0depzcrXwMHs2iyKmPByxNfO4I24hxr9DEusn2vlb-rxE9QV939IbrqBKLC423YIyXog8qZBnAc6ppSdT16lLrS7rzlqJXEJVkvScSwPoPbdWFvIDO1Ol0lVe-2bLuwKIyHyV1fL3L8uNOY__XNdeanaLk0_YEYUjD0Y99eQ0GCC3cZyb0WtDKnmcDjgD3pzjCy9BxpATs-bjOb4m8R-LCYcBifyvJbdDZxnEiFdR6b90vR7b9cbh7on-anh3bU6Qu_XAwRw3ZgsCpcmCAx6uJ413x3X3p0tNh3qP2rouNfvh8OIYXOdPsmgtrOP724O9XyI6A5qod8HYlqV1iE5v4fo0ijGgOR7bL4cmyeN91cnzb_Y3OT1r29bKAayTWfNKR4LHzj_2X0J4HezFzf1cESFGaoG_JEcSI3_g5Ml-ZRlsOGDP77mM1aApW3AUr9z5uDU2geLSj9wPLVLBF6JyoXY0pVvPpUlkrUTzCboFTEP_eTymR68imw52qGY0gpT8_ICsMLZvpL355XiFWuIPH6Uty8NGENcAGejyjOZMrfSxdQgqZahceQhmarg5wKvzSZdP2mRiu2hASrCK7CV3coQaoCwiYZhObpvzDyBxtD8-BELNAjcreZx-vstu4NQHecSAEjugaaRrYMYhC2C0RjqVarUZYt0_-BTc1mw6ABxsXu-fdYHPJhl8iMF7UXNRIuKczEzbDXaMSSX0PvlSdthyGwZA31jl48cn-mdfVXvp0ftv7Dr1qZL0B7VbKEt06JV1mrawSEhlpX7-CjR1dF8tN3UeVJp81OCXS5ZG8EZoqO6ubvyUmLJNlumqz2lUO-qFzLJw-ao-LigjPboHVC2LeO79FYwgC1uDxOtwjJ3sEt5sSin6NnOg_-ODQpq7Zyvo.fA_PPF6URq2kw1YErnbdxA"
]


In [54]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Track total progress
emails = df_match_linkedin['email'].tolist()
total_tasks = len(emails)

# Create a tqdm progress bar
progress_bar = tqdm(total=total_tasks, desc="Processing emails", unit="email")


for i_e in range(0, len(emails), 1500):
    with ThreadPoolExecutor(max_workers=1000) as executor:
        futures = [executor.submit(get_author_main, emails[i], tokens[i_e // 1500]) for i in range(i_e, min(i_e + 1500, len(emails)))]
        for future in as_completed(futures):
            future.result()
            progress_bar.update(1)


progress_bar.close()
print("All tasks completed.")


Processing emails: 100%|██████████| 299/299 [00:08<00:00, 33.25email/s]

All tasks completed.


#### Get linkedin's profile

In [61]:
df_result = pd.read_json('email_match_linkedin.json', lines=True)
df_result = df_result[df_result['status']==200]
df_result['response'] = df_result['response'].apply(lambda x: json.loads(x))
df_result['resultTemplate'] = df_result['response'].apply(lambda x:x.get('resultTemplate'))

In [64]:
df_linkedin_profile = df_result[df_result['resultTemplate']=='ExactMatch']
df_linkedin_profile

,time,token,email,status,response,resultTemplate
6299,2024-08-13 11:05:34,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,tranlamdong@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6300,2024-08-13 11:05:34,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,dangtuanqn@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6301,2024-08-13 11:05:34,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,tuanquoc230968@yahoo.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6302,2024-08-13 11:05:34,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,HOANGKIMHUE1954@YAHOO.COM,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6303,2024-08-13 11:05:34,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,VCCICORPORATION@GMAIL.COM,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
...,...,...,...,...,...,...
6593,2024-08-13 11:05:38,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,Kienviet.jsc@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6594,2024-08-13 11:05:38,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,caovuong@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6595,2024-08-13 11:05:38,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,buivutien@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch
6596,2024-08-13 11:05:38,Bearer eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjh...,leviet365@gmail.com,200,"{'resultTemplate': 'ExactMatch', 'bound': Fals...",ExactMatch


#### Sample linkedin profile

In [65]:
df_linkedin_profile['response'].sample(1).values[0]

{'resultTemplate': 'ExactMatch',
 'bound': False,
 'bindUrl': 'https://login.microsoftonline.com/2678b389-26e0-4e8a-b570-6a95ca4d8358/bind?Provider=LinkedIn&client_id=394866fc-eedb-4f01-8536-3ff84b16be2a&redirect_uri=https://loki.delve.office.com/linkedInAuthRedirect.aspx&login_hint=lexuanhoang120@tql3p.onmicrosoft.com&dualbind=true&dualbindpat=true&mkt=EN-US&external_app=Owa&dualbindmobile=True',
 'persons': [{'id': 'urn:li:person:DgEIHWKNbmi9w-LY6CPZfyzEn7T_iFs11Cmkavmnzac',
   'displayName': 'Hoang Do',
   'firstName': 'Hoang',
   'lastName': 'Do',
   'phoneNumbers': [],
   'headline': 'President of CPLATFORM',
   'companyName': 'CPlatform',
   'location': 'Vietnam',
   'photoUrl': 'https://media.licdn.com/dms/image/v2/D5603AQE-pKEGXTClsw/profile-displayphoto-shrink_400_400/profile-displayphoto-shrink_400_400/0/1695045683840?e=1723615200&v=beta&t=EBiA229QWJdg3B8UhALQZqLq2d0-0RP9nNsqi5KKJSU',
   'linkedInUrl': 'https://linkedin.com/in/hoang-do-430b98a7',
   'reportProfileUrl': 'https